In [1]:
'''
# Group 38 — Training notebook
# APPROACH EXPLANATION
# -------------------------------------------------------------------
#Model architecture and roadmap

EfficientNet-B3 pretrained on ImageNet, with the final classification head replaced by a Linear(1536 -> 2) layer -> binary output
    Input resolution: 384x384 (higher than the B3 default of 300x300 —
    better detail for subtle surface defects)

IMPORTANT key design decisions - 
   we use the provided COCO annotations (category_id == 22) to crop each image to the bottle body before passing it to the model ->
   this removes irrelevant background, letting the model focus on the surface

AMP (automatic mixed precision) — speeds up training
   Training augmentations -- RandomHorizontalFlip, RandomRotation(180),
   RandomAdjustSharpness, RandomAutocontrast 
   The 180 degrees rotation is intentional — bottles can appear inverted in captures!

 Validation strategy
---------------------------------------------------------------------------------
   80/20 stratified split (random_state=42), used to monitor overfitting
   Added Accuracy and F1-Score monitoring per epoch
'''

'\n# Group 38 — Training notebook\n# APPROACH EXPLANATION\n# -------------------------------------------------------------------\n#Model architecture and roadmap\n\nEfficientNet-B3 pretrained on ImageNet, with the final classification head replaced by a Linear(1536 -> 2) layer -> binary output\n    Input resolution: 384x384 (higher than the B3 default of 300x300 —\n    better detail for subtle surface defects)\n\nIMPORTANT key design decisions - \n   we use the provided COCO annotations (category_id == 22) to crop each image to the bottle body before passing it to the model ->\n   this removes irrelevant background, letting the model focus on the surface\n\nAMP (automatic mixed precision) — speeds up training\n   Training augmentations -- RandomHorizontalFlip, RandomRotation(180),\n   RandomAdjustSharpness, RandomAutocontrast \n   The 180 degrees rotation is intentional — bottles can appear inverted in captures!\n\n Validation strategy\n-------------------------------------------------

In [2]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from torchvision import models, transforms
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pandas as pd
import json

def roi_def(json_path):
    with open(json_path, 'r') as read_annotations:
        data = json.load(read_annotations)
        
    # Linking photo id and file name
    id_to_name = {img['id']: img['file_name'] for img in data['images']}
    roi_dict = {}
    
    for annot in data['annotations']:
        if annot['category_id'] == 22:
            img_name = id_to_name[annot['image_id']]
            roi_dict[img_name] = annot['bbox']      
    print(f"{len(roi_dict)} ROI to crop - found!")
    return roi_dict

In [3]:
#test transform -> eval notebook
train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.5),
    transforms.RandomAutocontrast(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
class BottleTrainDataset(Dataset):
    def __init__(self, dataframe, img_dir, roi_dict):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.roi_dict = roi_dict

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['image_id']
        label = self.df.iloc[idx]['target']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        
        #ROI crop
        if img_name in self.roi_dict:
            x, y, width, height = self.roi_dict[img_name]
            image = image.crop((x, y, x + width, y + height)) #cropping based on ROI
        return train_transform(image), label



In [5]:
# Config setup
TRAIN_CSV = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/train.csv'
TRAIN_DIR = '/kaggle/input/datasets/dreval/train-images/train_images'
TRAIN_JSON = '/kaggle/input/competitions/1st-krones-vision-ai-challenge/train_annotations.json'

BATCH_SIZE = 32
EPOCHS = 12
L_RATE = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_full = pd.read_csv(TRAIN_CSV)
train, valid = train_test_split(dataset_full, test_size=0.2, random_state=42)

print("Loading data") 
train_roi_dict = roi_def(TRAIN_JSON)
train_loader = DataLoader(BottleTrainDataset(train, TRAIN_DIR, train_roi_dict), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
#using the same transform for validation during testing
valid_loader = DataLoader(BottleTrainDataset(valid, TRAIN_DIR, train_roi_dict), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Loading data
35342 ROI to crop - found!


In [6]:
# Init model
print("Init model - EFFICIENTNET-B3")
model = models.efficientnet_b3(weights='IMAGENET1K_V1')
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)
model = model.to(DEVICE)

# Loss, optimizer, scheduler
crit = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=L_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Init model - EFFICIENTNET-B3
Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth


100%|██████████| 47.2M/47.2M [00:00<00:00, 163MB/s]


In [7]:
'''
#training loop
Initializing GradScaler for Automatic Mixed Precision (AMP)
This scales the loss to prevent underflow when using float16 gradients-> allowing for faster training + significantly lower GPU memory usage
'''
scaler = torch.amp.GradScaler('cuda') #faster training preventing out-of-memory errors
best_valid_loss = 999999.0 #it will severely drop based on iterations through epochs -> to save at the end the ebst model

print("Training Started")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = crit(outputs, labels)

        #backward pass- scaling the loss, gradients computing + weights update (backprop)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        
    train_loss = running_loss / len(train_loader) #average validation loss
    
    # Validation phase - preparations
    model.eval()
    valid_loss = 0.0
    all_preds = [] 
    all_labels = []

    #disabling gradient calculation to save memory and speed up inference
    with torch.no_grad():
        for valid_imgs, valid_labels in valid_loader:
            valid_imgs, valid_labels = valid_imgs.to(DEVICE), valid_labels.to(DEVICE)
            
            with torch.amp.autocast('cuda'):
                valid_outputs = model(valid_imgs)
                v_loss = crit(valid_outputs, valid_labels)

            valid_loss += v_loss.item()
            
            # taking the prediction 
            #extracting the predicted class index (highest probability)
            _, preds = torch.max(valid_outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(valid_labels.cpu().numpy())

    valid_loss = valid_loss / len(valid_loader) #average validation loss
    
    #metrixcs calc
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} | LR: {current_lr:.6f}")

    if valid_loss < best_valid_loss:
        print(f" - Validation loss improved ({best_valid_loss:.4f} -> {valid_loss:.4f}). Saving model!")
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pth')
        
    scheduler.step()

print("Training finished! Best model weights saved as best_model.pth")

Training Started
Epoch 1/12 | Train Loss: 0.2021 | Valid Loss: 0.1586 | Val Acc: 0.9311 | Val F1: 0.9299 | LR: 0.000500
 - Validation loss improved (999999.0000 -> 0.1586). Saving model!
Epoch 2/12 | Train Loss: 0.1478 | Valid Loss: 0.1302 | Val Acc: 0.9464 | Val F1: 0.9446 | LR: 0.000491
 - Validation loss improved (0.1586 -> 0.1302). Saving model!
Epoch 3/12 | Train Loss: 0.1326 | Valid Loss: 0.1129 | Val Acc: 0.9542 | Val F1: 0.9528 | LR: 0.000467
 - Validation loss improved (0.1302 -> 0.1129). Saving model!
Epoch 4/12 | Train Loss: 0.1215 | Valid Loss: 0.1479 | Val Acc: 0.9421 | Val F1: 0.9410 | LR: 0.000427
Epoch 5/12 | Train Loss: 0.1124 | Valid Loss: 0.1190 | Val Acc: 0.9523 | Val F1: 0.9510 | LR: 0.000375
Epoch 6/12 | Train Loss: 0.1031 | Valid Loss: 0.1125 | Val Acc: 0.9566 | Val F1: 0.9550 | LR: 0.000315
 - Validation loss improved (0.1129 -> 0.1125). Saving model!
Epoch 7/12 | Train Loss: 0.0977 | Valid Loss: 0.1097 | Val Acc: 0.9581 | Val F1: 0.9568 | LR: 0.000250
 - Valida